# 02. Pareto Efficiency Frontier & 12-Month Inference TCO Model
**Objective:** Evaluate **Capability-per-Dollar** and **Capability-per-Ton-CO2** across frontier models, and model full lifecycle TCO (Pre-Training CapEx/OpEx + 12-Month Serving Inference).

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/02_gold_exports/models_enriched_gold.csv')
pareto_df = df.dropna(subset=['Composite_Capability_Index', 'total_cost_usd']).copy()
pareto_df = pareto_df[(pareto_df['Composite_Capability_Index'] > 0) & (pareto_df['total_cost_usd'] > 0)]

# Compute Pareto Frontier (Minimize Cost, Maximize Composite Capability Index)
pareto_df = pareto_df.sort_values('total_cost_usd')
frontier_rows = []
max_cap = -1.0
for _, row in pareto_df.iterrows():
    if row['Composite_Capability_Index'] > max_cap:
        frontier_rows.append(row)
        max_cap = row['Composite_Capability_Index']
frontier = pd.DataFrame(frontier_rows)
print(f'Models evaluated: {len(pareto_df)} | Pareto-optimal models: {len(frontier)}')
frontier[['model_name', 'organization', 'publication_year', 'total_cost_usd', 'Composite_Capability_Index', 'usd_per_capability_point', 'co2_tons_per_capability_point']]

In [ ]:
# Chart 9: Frontier Capability vs. Training Cost Pareto Curve
fig = px.scatter(
    pareto_df, x='total_cost_usd', y='Composite_Capability_Index',
    color='organization', hover_name='model_name',
    size=pareto_df['facility_total_energy_mwh'].fillna(pareto_df['facility_total_energy_mwh'].median()), log_x=True,
    title='AI Economics Pareto Frontier: Capability Index vs. Training Cost (USD)',
    labels={'total_cost_usd': 'Estimated Training Cost (USD, Log Scale)', 'Composite_Capability_Index': 'Composite Frontier Capability Index (0-100)'},
    template='plotly_white', width=1050, height=580
)
fig.add_trace(go.Scatter(
    x=frontier['total_cost_usd'], y=frontier['Composite_Capability_Index'],
    mode='lines+markers+text', text=frontier['model_name'], textposition='top left',
    line=dict(color='#DC2626', width=2.5, dash='dash'),
    marker=dict(size=10, color='#DC2626', symbol='diamond'),
    name='Pareto Efficiency Frontier'
))
fig.write_image('../assets/charts/chart9_pareto_frontier.png', scale=2)
fig.show()

In [ ]:
# 12-Month Lifecycle TCO Simulator (Pre-Training + Production Inference Serving)
def simulate_12m_tco(df_models, daily_requests=10_000_000, avg_output_tokens=500, watts_per_1k_tokens=0.45):
    sim = df_models.dropna(subset=['total_cost_usd', 'facility_total_energy_mwh']).copy()
    annual_tokens_k = (daily_requests * avg_output_tokens * 365) / 1000.0
    # Scale inference energy modestly with model parameter count (normalized to 70B baseline)
    param_factor = np.log10(sim['parameters'].fillna(7e10).clip(lower=1e9)) / np.log10(7e10)
    sim['annual_inference_mwh'] = (annual_tokens_k * watts_per_1k_tokens * param_factor) / 1e6 * sim['facility_pue']
    sim['annual_inference_electricity_usd'] = sim['annual_inference_mwh'] * sim['tariff_usd_mwh']
    sim['annual_inference_co2_tons'] = (sim['annual_inference_mwh'] * 1000.0 * sim['carbon_gco2_kwh']) / 1e6
    sim['inference_to_training_energy_ratio'] = sim['annual_inference_mwh'] / sim['facility_total_energy_mwh']
    return sim[['model_name', 'organization', 'facility_total_energy_mwh', 'annual_inference_mwh', 'inference_to_training_energy_ratio', 'total_cost_usd', 'annual_inference_electricity_usd']]

tco_summary = simulate_12m_tco(df)
tco_summary.sort_values('facility_total_energy_mwh', ascending=False).head(10)